# EE344 - Winter26 - HW 1

In this notebook, we will:
- Generate **two synthetic regression datasets**:
  1. Linear relationship
  2. Cubic polynomial relationship (degree 3)
- Fit model to each dataset using a **Linear Regressor**:
  - Linear Regression
  - Polynomial Regression (degree 3) using polynomial feature expansion
- Evaluate models using a **train/test split** (70% train, 30% test)

We report:
- **RMSE**
- **$R^2$**



In [1]:
# ============================================================
# Imports
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt

## 1) Load the dataset and inspect basic information

In this section, we load the "fuel_consumption_hp.csv" dataset into a pandas DataFrame and perform a **basic sanity check** before building any models.
This is for part 1 only.

### What this code does
- **Loads the CSV file** into a pandas DataFrame (`df`)
- Prints the **shape** of the dataset:
  - number of rows = number of samples (students)
  - number of columns = number of features (variables)
- Prints the **column names** to understand what information is available
- Displays the **first few rows** using `head()` to preview the data format and values
- Shows **summary statistics** using `describe()`:
  - for numeric columns: mean, standard deviation, min/max, quartiles, etc.
  - for non-numeric columns: count, unique values, most common value, etc.
- Checks for **missing values** in each column


In [4]:
df = pd.read_csv("fuel_consumption_hp.csv")

# ============================================================
# Load dataset
# ============================================================

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

print("\nSummary statistics:")
display(df.describe(include="all"))

print("\nMissing values per column:")
display(df.isna().sum())

Shape: (100, 2)

Columns:
['Horse Power', 'Fuel Economy (MPG)']


,Horse Power,Fuel Economy (MPG)
0,118.770799,29.344195
1,176.326567,24.695934
2,219.262465,23.952010
3,187.310009,23.384546
4,218.594340,23.426739



Summary statistics:


,Horse Power,Fuel Economy (MPG)
count,100.000000,100.000000
mean,213.676190,23.178501
std,62.061726,4.701666
min,50.000000,10.000000
25%,174.996514,20.439516
50%,218.928402,23.143192
75%,251.706476,26.089933
max,350.000000,35.000000



Missing values per column:


Horse Power           0
Fuel Economy (MPG)    0
dtype: int64

## 2) Utility functions

In below, we will run regression experiments on 2 different datasets.  
To keep our notebook clean and avoid copy-pasting the same code many times, we define a few **helper functions**.

These functions handle the most common steps in any supervised learning workflow:
1. Preparing features **X** and target **y**
2. Splitting data into **train** and **test**
3. Training linear and polynomial regression models
4. Computing evaluation metrics


### What each function does

- **`prepare_xy(df_in)`**  
  Removes rows with missing values and splits the dataset into:
  - **X** = input features (all columns except the target)  
  - **y** = target variable (here: `Performance Index`)

- **`split_data(X, y)`**  
  Performs a **70% / 30% train-test split** using a fixed `random_state` so that results are reproducible.

- **`compute_metrics(y_true, y_pred)`**  
  Computes three standard regression evaluation metrics:
  - **MSE (Mean Squared Error):** penalizes large errors more strongly  
  - **MAE (Mean Absolute Error):** average absolute prediction error  
  - **R² (Coefficient of Determination):** measures how well the model explains the variance in the data

- **`run_models_and_evaluate(...)`**  
  This is the main driver function that runs everything for a given scenario:
  - Trains **Linear Regression** and **Polynomial Regression (degrees 2, 3, 4)**
  - Evaluates **train and test** performance using MSE, MAE, and R²
  - Prints fitted equation (top terms)
  - Generates test-set scatter plots
  - Returns a clean results table for easy comparison

---

✅ After this section, the rest of the notebook becomes much shorter and easier to read, because each scenario can reuse these helper functions.


In [11]:
TARGET_COL = "Horse Power"


def prepare_xy(df_in, target_col = TARGET_COL):
    """Drop missing rows, split into X and y."""
    df_clean = df_in.dropna().copy()
    X = df_clean.drop(columns=[target_col])
    y = df_clean[target_col]
    return X, y

def split_data(X, y, test_size=0.30, random_state=42):
    """70/30 random train-test split."""
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


def compute_metrics(y_true, y_pred):
    """Return MSE, MAE, R^2."""
    return {
        "MSE": mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R^2": r2_score(y_true, y_pred),
    }

def run_models_and_evaluate(df_in, scenario_name, target_col, degrees=(1, 2, 3, 4),
                             test_size=0.30, random_state=42,):
    """Train/evaluate linear (deg=1) + polynomial regression models.

    Returns a DataFrame of metrics.
    Also prints fitted equations and scatter plots (test set) for each model.
    """
    X, y = prepare_xy(df_in, target_col)
    X_train, X_test, y_train, y_test = split_data(X, y, test_size=test_size, random_state=random_state)

    rows = []

    for deg in degrees:
        if deg == 1:
            model = LinearRegression()
            model_name = "Linear Regression"
        else:
            model = Pipeline([
                ("poly", PolynomialFeatures(degree=deg, include_bias=False)),
                ("lr", LinearRegression())
            ])
            model_name = f"Polynomial Regression (degree={deg})"

        # Fit model
        model.fit(X_train, y_train)

        # Predict
        yhat_train = model.predict(X_train)
        yhat_test  = model.predict(X_test)

        # Metrics
        train_m = compute_metrics(y_train, yhat_train)
        test_m  = compute_metrics(y_test, yhat_test)

    

        rows.append({
            "Scenario": scenario_name,
            "Model": model_name,
            "Train MSE": train_m["MSE"],
            "Train MAE": train_m["MAE"],
            "Train R^2": train_m["R^2"],
            "Test MSE": test_m["MSE"],
            "Test MAE": test_m["MAE"],
            "Test R^2": test_m["R^2"],
            "Train size": len(X_train),
            "Test size": len(X_test),
        })

    return pd.DataFrame(rows)


## Part 1 — Fuel Consumption → Horsepower Prediction

Train and evaluate models using linear regression and polynomial regression (degree 2, 3, 4).
Reports **Train and Test** metrics:
  - **MSE**, **MAE**, and **R²**

In [12]:

TARGET_COL = "Horse Power"
target_col = TARGET_COL
results_A = run_models_and_evaluate(
    df,
    scenario_name="Fuel Consumption → Horsepower",
    target_col=TARGET_COL,
    degrees=(1, 2, 3, 4),
)

display(results_A)

,Scenario,Model,Train MSE,Train MAE,Train R^2,Test MSE,Test MAE,Test R^2,Train size,Test size
0,Fuel Consumption → Horsepower,Linear Regression,357.699180,16.061689,0.906320,318.561087,14.940628,0.912561,70,30
1,Fuel Consumption → Horsepower,Polynomial Regression (degree=2),350.879731,15.995824,0.908106,331.105434,15.148330,0.909118,70,30
2,Fuel Consumption → Horsepower,Polynomial Regression (degree=3),345.108668,15.746762,0.909618,318.404012,14.764973,0.912604,70,30
3,Fuel Consumption → Horsepower,Polynomial Regression (degree=4),339.700171,15.508465,0.911034,313.798757,14.735471,0.913868,70,30


# Part 1 Discussion and interpretation(1.5)
## Which model performs best on the test set and why?
The Polynomial Regression with degree = 4 performs best on the test set.
It has the lowest Test MSE (313.80) and lowest Test MAE (14.74), as well as the highest Test R² (0.9139) among all models.
Compared to linear regression (Test R² = 0.9126), there is little and consistent improvement across all three metrics, indicating slightly better generalization rather than just random.
## Does increasing polynomial degree always improve performance? If not, explain what you observe.
Increasing the polynomial degree does not always monotonically improve performance.
Degree 2 has worse Test MSE (331.11) and lower Test R² (0.9091) than the linear model. Degree 3 has about the same test performance of linear regression. Degree 4 has the best test results.
This suggest that 1 degree increase in model does not guarantee better performence compared to previous model.
## If a model performs unexpectedly poorly (e.g., low R2 or large test error), propose at least two plausible reasons:
Reason 1 is that degree 2 may be too small to model the true relationship between fuel consumption and horsepower, resulting in higher test error despite slightly improved training error.
Reason 2: 70 training and 30 test samples are too small, this may have small variations in the data and can disproportionately affect polynomial models, especially lower-degree ones.

## Part 2. Load the dataset and inspect basic information

In this section, we load the "electricity_consumption_based_weather_dataset.csv" dataset into a pandas DataFrame and perform a **basic sanity check** before building any models.

### What this code does
- **Loads the CSV file** into a pandas DataFrame (`df`)
- Prints the **shape** of the dataset:
  - number of rows = number of samples
  - number of columns = number of features
- Prints the **column names** to understand what information is available
- Displays the **first few rows** using `head()` to preview the data format and values
- Shows **summary statistics** using `describe()`:
  - for numeric columns: mean, standard deviation, min/max, quartiles, etc.
  - for non-numeric columns: count, unique values, most common value, etc.
- Checks for **missing values** in each column


In [ ]:

df2 = pd.read_csv("electricity_consumption_based_weather_dataset.csv")

print("Shape", df2.shape)
print("\nColumns:")
print(df2.columns.tolist())

display(df2.head())

print("\nSummary Statistics:")
print(df2.describe(include="all"))

print("\nMissing values per column:")
print(df2.isna().sum())

Shape (1433, 6)

Columns:
['date', 'AWND', 'PRCP', 'TMAX', 'TMIN', 'daily_consumption']


,date,AWND,PRCP,TMAX,TMIN,daily_consumption
0,2006-12-16,2.5,0.0,10.6,5.0,1209.176
1,2006-12-17,2.6,0.0,13.3,5.6,3390.460
2,2006-12-18,2.4,0.0,15.0,6.7,2203.826
3,2006-12-19,2.4,0.0,7.2,2.2,1666.194
4,2006-12-20,2.4,0.0,7.2,1.1,2225.748



Summary Statistics:
              date         AWND         PRCP         TMAX         TMIN  \
count         1433  1418.000000  1433.000000  1433.000000  1433.000000   
unique        1433          NaN          NaN          NaN          NaN   
top     2006-12-16          NaN          NaN          NaN          NaN   
freq             1          NaN          NaN          NaN          NaN   
mean           NaN     2.642313     3.800488    17.187509     9.141242   
std            NaN     1.140021    10.973436    10.136415     9.028417   
min            NaN     0.000000     0.000000    -8.900000   -14.400000   
25%            NaN     1.800000     0.000000     8.900000     2.200000   
50%            NaN     2.400000     0.000000    17.800000     9.400000   
75%            NaN     3.300000     1.300000    26.100000    17.200000   
max            NaN    10.200000   192.300000    39.400000    27.200000   

        daily_consumption  
count         1433.000000  
unique                NaN  
top   

## Part 2 — Fuel Consumption → Horsepower Prediction

Train and evaluate models using linear regression and polynomial regression (degree 2, 3, 4).
Reports **Train and Test** metrics:
  - **MSE**, **MAE**, and **R²**

In [10]:
EXTRA_COL = "date"
TARGET_COL = "daily_consumption"
df2_no_date = df2.drop(columns=[EXTRA_COL]).copy()
display(df2_no_date.head())

target_col=TARGET_COL
results_part2 = run_models_and_evaluate(
    df2_no_date,
    scenario_name="daily_consumption prediction",
    target_col=TARGET_COL,
    degrees=(1, 2, 3, 4),
)

display(results_part2)


,AWND,PRCP,TMAX,TMIN,daily_consumption
0,2.5,0.0,10.6,5.0,1209.176
1,2.6,0.0,13.3,5.6,3390.460
2,2.4,0.0,15.0,6.7,2203.826
3,2.4,0.0,7.2,2.2,1666.194
4,2.4,0.0,7.2,1.1,2225.748



Scenario: daily_consumption prediction
Model: Linear Regression

Scenario: daily_consumption prediction
Model: Polynomial Regression (degree=2)

Scenario: daily_consumption prediction
Model: Polynomial Regression (degree=3)

Scenario: daily_consumption prediction
Model: Polynomial Regression (degree=4)


,Scenario,Model,Train MSE,Train MAE,Train R^2,Test MSE,Test MAE,Test R^2,Train size,Test size
0,daily_consumption prediction,Linear Regression,272403.396174,384.465016,0.276000,2.481258e+05,375.404537,0.299333,992,426
1,daily_consumption prediction,Polynomial Regression (degree=2),264765.769932,379.648753,0.296300,2.552685e+05,379.039083,0.279163,992,426
2,daily_consumption prediction,Polynomial Regression (degree=3),259249.534870,375.952901,0.310961,2.656237e+05,385.235167,0.249922,992,426
3,daily_consumption prediction,Polynomial Regression (degree=4),251909.339001,372.116566,0.330470,1.215149e+07,578.642201,-33.313844,992,426


# Part 2 Discussion and interpretation(2.5)
## Which model generalizes best (best test performance), and what does that tell you about the relationship between weather and electricity usage?
Linear Regression generalizes best on the test set, achieving the highest Test R² (0.299) and a lowest Test MSE (2.48 × 10⁵).
Although polynomial models achieve better training performance, their test performance is worse, R² is very low acrross all 4 models, indicating that the relationship between weather variables and daily electricity consumption has weak relashionsh and dominated by noise or unmodeled factors. The test R² across all models suggests that weather alone explains only a little of the variance in electricity usage.
## Do polynomial models improve the fit compared to linear regression? If yes, why might electricity consumption have nonlinear dependence on weather?
Polynomial models do improve the fit, with Train R² increasing from 0.276 (linear) to 0.330 (degree 4) and Train MSE and Train MAE decreasing steadily. This suggests that electricity consumption may have nonlinear dependence on weather. 
## If higher-degree models perform worse on the test set, explain this behavior using evidence frommetrics (e.g., train error decreases but test error 
## increases).
Higher-degree polynomial models may be overfitting. From linear model to polynomial degree 4 model, Training error decreases monotonically (Train MSE drops from 2.72 × 10⁵ to 2.52 × 10⁵), while Test error increases steadly, especially for degree 4, where Test MSE explodes to 1.21 × 10⁷ and Test R² becomes strongly negative (-33.31) which means prediction itself is worse. Higher-degree models might fit noise and random patterns in the training data, resulting in worse generalization.
## If none of the models achieve good test performance, provide at least two reasons supported by your outputs (e.g., limited feature set, high noise 
## unmodeled drivers such as occupancy/behavior, seasonal effects).
Reason 1: Among all variables that affect electricity consumption, weather alone are insufficient to predict electricity consumption accurately. There might be other variables that has highr relashionship such as occupancy patterns, appliance usage, building characteristics.
Reason 2: Daily electricity usage is more related to human than weather, which introduces significant randomness that cannot be explained by weather data alone.